# Statistics for Data Analysts/Engineers — Module 1: Descriptive Statistics

Welcome to the statistics course! If you've been through our Python and SQL courses, you
already have the tools to pull data and write code — now we'll learn to draw conclusions
from that data. Statistics splits into two big branches: **descriptive** (how to
concisely summarize the data you have) and **inferential** (how to draw conclusions about
a whole population from a sample — we'll get there starting in Module 5). This module is
the foundation: without being comfortable describing data, it's hard to test it later.

## Table of contents
1. [What is descriptive statistics?](#sec1)
2. [Measures of central tendency: mean, median, mode](#sec2)
3. [Measures of spread: variance and standard deviation](#sec3)
4. [Percentiles, quartiles, and IQR](#sec4)
5. [Detecting outliers](#sec5)
6. [`describe()` — a quick pandas summary](#sec6)
7. [Summary and exercises](#sec7)


Every module in this course starts by generating the same synthetic dataset -- thanks to a fixed random seed (`np.random.default_rng(42)`), the numbers always come out identical, so you can open any module independently of the others, even after restarting the kernel.

Throughout this module (and most of the following ones) we'll work with the same synthetic `customers` dataset -- simulated data from an online shop. The `monthly_spend` column will be our main example, because it has a realistic, right-skewed shape (most customers spend little, a handful spend a lot).

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

city = rng.choice(
    ["New York", "Chicago", "Austin", "Seattle", "Denver"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["North", "South", "East", "West"], size=n)
age = rng.integers(18, 70, size=n)

# monthly spend: right-skewed distribution (typical for spending data)
monthly_spend = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satisfaction correlated with spend + a region effect (deliberately different) + random
# noise, clipped to a 1-10 scale -- this gives Module 10 (ANOVA) a real between-group gap
region_effect = {"North": 0.0, "South": -0.5, "East": 0.3, "West": 0.8}
satisfaction = np.clip(
    monthly_spend / 45
    + np.array([region_effect[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# A/B campaign group -- with a REAL injected effect on conversion (group B better)
group = rng.choice(["A", "B"], size=n)
conversion_prob = np.where(group == "A", 0.10, 0.20)
converted = rng.binomial(1, conversion_prob)

# spend before/after a loyalty-program rollout (paired data)
spend_before = rng.normal(220, 40, size=n).round(2)
program_effect = rng.normal(15, 10, size=n)
spend_after = (spend_before + program_effect).round(2)

customers = pd.DataFrame({
    "customer_id": np.arange(1, n + 1),
    "city": city,
    "region": region,
    "age": age,
    "monthly_spend": monthly_spend,
    "satisfaction": satisfaction,
    "campaign_group": group,
    "converted": converted,
    "spend_before": spend_before,
    "spend_after": spend_after,
})
customers.head()


<a id="sec1"></a>
## 1. What is descriptive statistics?

**Descriptive statistics** answers the question: "how do I concisely describe a set of
numbers so someone else (or you, a week from now) understands its shape without looking
at all 400 rows?". Instead of pasting the entire `monthly_spend` column, we say something
like "on average customers spend about $190 a month, but the spread is quite wide, from a
few dozen dollars to over $500" -- that's exactly the distillation of data down to a
handful of numbers we'll learn to compute in this module.

In [ ]:
customers["monthly_spend"].describe()


This one line of `pandas` already shows most of what we'll learn to compute by hand in
this module: the observation count, mean, standard deviation, minimum, quartiles, and
maximum. Let's break it down piece by piece.

<a id="sec2"></a>
## 2. Measures of central tendency: mean, median, mode

**Measures of central tendency** try to answer: "what is the typical, central value in
this dataset?". There are three basic ones:

- **Mean** — the sum of all values divided by their count. Sensitive to outliers -- a
  single very large value can pull it strongly.
- **Median** — the middle value once the data is sorted (or the average of the two middle
  values when the count is even). Robust to outliers.
- **Mode** — the most frequently occurring value. Mostly meaningful for discrete/
  categorical data, less often for continuous measurements.

In [ ]:
import numpy as np
from scipy import stats

spend = customers["monthly_spend"]

mean_val = spend.mean()
median_val = spend.median()
mode_val = stats.mode(spend, keepdims=False)

print(f"Mean: {mean_val:.2f}")
print(f"Median: {median_val:.2f}")
print(f"Mode: {mode_val.mode:.2f} (occurred {mode_val.count} times)")


> ⚠️ **Mean ≠ median when the distribution is skewed**
>
> Notice the mean comes out noticeably higher than the median. That's a sign the distribution is **right-skewed** (it has a "tail" toward large values) -- a few customers spending a lot pull the mean upward without changing the median. We'll look at this visually in Module 2. Rule of thumb: when data is skewed or has outliers, **the median better describes the "typical" customer** than the mean.

> 📊 **Median income, not mean income**
>
> This is exactly why national statistics usually report the **median** income rather than the mean -- the mean is strongly pulled up by a small group of very high earners and doesn't reflect what a "typical" person actually earns.

<a id="sec3"></a>
## 3. Measures of spread: variance and standard deviation

A central-tendency measure alone says nothing about how spread out the data is. Two
datasets can share the same mean and look completely different -- one clustered tightly
around the mean, the other widely scattered. **Measures of spread** capture this:

- **Range** — the difference between the maximum and minimum. Simple, but very sensitive
  to a single extreme value.
- **Variance** — the average of the squared deviations from the mean. Its unit is
  squared (e.g. "dollars squared"), which makes it hard to interpret directly.
- **Standard deviation** — the square root of the variance. It returns to the original
  unit (dollars) and is the most commonly used measure of spread.

In [ ]:
range_val = spend.max() - spend.min()
variance_val = spend.var(ddof=1)
std_val = spend.std(ddof=1)

print(f"Range: {range_val:.2f}")
print(f"Variance: {variance_val:.2f}")
print(f"Standard deviation: {std_val:.2f}")


> ⚠️ **ddof=1: sample vs. population**
>
> The `ddof` argument ("delta degrees of freedom") decides whether you're computing variance for the **entire population** (`ddof=0`, divide by `n`) or for a **sample** representing a larger population (`ddof=1`, divide by `n-1`). Dividing by `n-1` (the so-called Bessel correction) compensates for the fact that a sample inherently tends to *underestimate* the population's true spread. `pandas` defaults to `ddof=1` -- in practice you're almost always working with a sample, so that's a sensible default.

<a id="sec4"></a>
## 4. Percentiles, quartiles, and IQR

A **percentile** *p* is the value below which *p*% of observations fall. The 50th
percentile is exactly the median. Especially useful are the **quartiles**: Q1 (25th
percentile), Q2 (50th percentile = median), and Q3 (75th percentile) -- they split the
data into four equally-sized (by count) parts.

The **interquartile range (IQR)** is `Q3 - Q1`: the spread of the "middle" 50% of the
data, far more robust to outliers than a plain range.

In [ ]:
q1 = spend.quantile(0.25)
q2 = spend.quantile(0.50)
q3 = spend.quantile(0.75)
iqr = q3 - q1

print(f"Q1 (25%): {q1:.2f}")
print(f"Q2 / median (50%): {q2:.2f}")
print(f"Q3 (75%): {q3:.2f}")
print(f"IQR: {iqr:.2f}")


`quantile()` accepts any value between 0 and 1, not just quartiles -- e.g.
`spend.quantile(0.90)` gives you the value below which 90% of customers fall (handy for
questions like "at what amount does the top 10% of spenders start?").

<a id="sec5"></a>
## 5. Detecting outliers

An **outlier** is a value that stands out significantly from the rest of the data. The
most common, simple detection method relies on the IQR from the previous section:
anything below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR` is treated as a potential outlier.
The `1.5` multiplier is a convention (the exact same one the default box plot uses -- you'll
see it in Module 2), not a law of nature.

In [ ]:
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = customers[
    (customers["monthly_spend"] < lower_bound) | (customers["monthly_spend"] > upper_bound)
]
print(f"Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Number of outliers: {len(outliers)}")
outliers[["customer_id", "monthly_spend"]].sort_values("monthly_spend", ascending=False)


> 📊 **An outlier isn't always a mistake**
>
> It's tempting to assume an outlier is a data error (a typo, a faulty sensor) -- sometimes it is, and then you should fix or remove it. But sometimes it's simply a genuine, rare case (a VIP customer who really does spend a lot), and removing it would distort the picture of reality. The IQR rule tells you **what to check manually**, not what to automatically discard.

<a id="sec6"></a>
## 6. `describe()` — a quick pandas summary

Let's revisit the cell from section 1 -- now you understand every number it returns:

In [ ]:
customers["monthly_spend"].describe()


`count` is the number of observations, `mean`/`std` are the mean and standard deviation
from sections 2-3, and `min`/`25%`/`50%`/`75%`/`max` are the minimum, quartiles, and
maximum from section 4. Calling `describe()` on the whole DataFrame
(`customers.describe()`) computes the same statistics for every numeric column at once,
and `customers.describe(include="object")` shows a different set of statistics (including
the most frequent value) for text/categorical columns like `city`.

In [ ]:
customers.describe(include="object")


<a id="sec7"></a>
## 7. Summary and exercises

In this module we covered:
- measures of central tendency: mean, median, mode -- and when the mean is misleading,
- measures of spread: range, variance, standard deviation (and the `ddof=0` vs `ddof=1`
  distinction),
- percentiles, quartiles, and IQR,
- a simple IQR rule for detecting outliers,
- `describe()` as a quick, ready-made overview of everything at once.

> 📝 **Exercise 1: Age statistics**
>
> Compute the mean, median, and standard deviation of the `age` column. Are the mean and median close to each other? What does that suggest about the shape of the age distribution (compared to spend in section 2)?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
print(customers["age"].mean())
print(customers["age"].median())
print(customers["age"].std(ddof=1))
```

Age is generated from a uniform distribution (rng.integers), so the mean and median should come out very close to each other -- unlike spend, age isn't skewed.
</details>

> 📝 **Exercise 2: IQR and outliers for satisfaction**
>
> Compute Q1, Q3, and IQR for the `satisfaction` column, then find the rows flagged as outliers by the `1.5 * IQR` rule. How many are there?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
q1_s = customers["satisfaction"].quantile(0.25)
q3_s = customers["satisfaction"].quantile(0.75)
iqr_s = q3_s - q1_s
lower_s = q1_s - 1.5 * iqr_s
upper_s = q3_s + 1.5 * iqr_s

outliers_s = customers[
    (customers["satisfaction"] < lower_s) | (customers["satisfaction"] > upper_s)
]
len(outliers_s)
```
</details>

> 📝 **Exercise 3: Statistics by city**
>
> Use `groupby("city")["monthly_spend"]` together with `.agg([...])` (from the Python/pandas course) to compute the mean, median, and standard deviation of spend separately for each city.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
customers.groupby("city")["monthly_spend"].agg(["mean", "median", "std"])
```
</details>

> 📝 **Exercise 4: Top 10% of customers**
>
> Find the spend threshold above which the top 10% of customers fall (hint: `quantile(0.90)`), then return a DataFrame of those customers sorted by spend descending.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
top10_threshold = customers["monthly_spend"].quantile(0.90)
top_customers = customers[customers["monthly_spend"] > top10_threshold]
top_customers.sort_values("monthly_spend", ascending=False)
```
</details>

> 🔥 **Challenge: your own summary function**
>
> Write a function `summarize(column)` that takes a numeric Series and returns a dictionary with keys: `mean`, `median`, `std`, `iqr`, `outlier_count` (computed with the rule from section 5). Test it on `monthly_spend` and on `age`, and compare the results.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
def summarize(column):
    q1 = column.quantile(0.25)
    q3 = column.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_count = ((column < lower) | (column > upper)).sum()
    return {
        "mean": column.mean(),
        "median": column.median(),
        "std": column.std(ddof=1),
        "iqr": iqr,
        "outlier_count": outlier_count,
    }


print(summarize(customers["monthly_spend"]))
print(summarize(customers["age"]))
```
</details>

## What's next?

In **Module 2** we'll move from numbers to pictures: histograms, box plots, and density
plots will show exactly what you already know from this module (skewness, IQR, outliers),
but in a form you can "take in at a glance" instead of reading a column of numbers.